# 🎵 AudioCraft (MusicGen) on Google Colab

MetaのMusicGenを使って、テキストプロンプトから音楽を生成するノートブックです。

| モデル | VRAM目安 | 生成速度 | 品質 |
|--------|---------|---------|------|
| `musicgen-small` | 約4GB | 速い | 標準 |
| `musicgen-medium` | 約8GB | 普通 | 高品質 |

> **推奨**: ランタイム → ランタイムのタイプを変更 → **T4 GPU** を選択してから実行してください。

## Step 1: 環境構築

必要なライブラリをインストールし、Google ドライブをマウントします。

> **初回のみ**: `audiocraft` のインストールに約1〜2分かかります。

In [ ]:
# ── Step 1: 環境構築 ──────────────────────────────────────────

# audiocraft は torch / torchaudio と依存関係があるため順序に注意
!pip install -q audiocraft
!pip install -q soundfile ipywidgets

import os
import torch

# Google ドライブのマウント
from google.colab import drive
drive.mount('/content/drive')

# 保存先ディレクトリを作成
DRIVE_MUSIC_DIR = '/content/drive/MyDrive/AI_Music'
LOCAL_MUSIC_DIR = '/content/music_output'
os.makedirs(DRIVE_MUSIC_DIR, exist_ok=True)
os.makedirs(LOCAL_MUSIC_DIR, exist_ok=True)

print('✅ ライブラリのインストール完了')
print(f'📁 ローカル保存先  : {LOCAL_MUSIC_DIR}')
print(f'📁 Drive 保存先    : {DRIVE_MUSIC_DIR}')

## Step 2: モデルのロード

`MODEL_SIZE` を `"small"` または `"medium"` に切り替えてください。

- **small** : 約1.5GB ダウンロード・T4 GPU で動作
- **medium**: 約3GB ダウンロード・A100 GPU 推奨

初回ダウンロードはキャッシュされるため、2回目以降は高速です。

In [ ]:
# ── Step 2: モデルのロード ────────────────────────────────────

from audiocraft.models import MusicGen

# ★ここを変更: "small" or "medium"
MODEL_SIZE = 'small'

MODEL_ID = f'facebook/musicgen-{MODEL_SIZE}'

# GPU / CPU 自動判定
if torch.cuda.is_available():
    device   = 'cuda'
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU 検出: {gpu_name}  (VRAM: {vram_gb:.1f} GB)')
    if MODEL_SIZE == 'medium' and vram_gb < 7:
        print('⚠️  medium モデルには 8GB 以上の VRAM を推奨します。')
        print('   small に変更するか、ランタイムを A100 に切り替えてください。')
else:
    device = 'cpu'
    print('⚠️  GPU が見つかりません。CPU で実行します（生成に数分かかります）')

# モデルロード（既にロード済みの場合はスキップ）
if 'musicgen_model' not in globals() or getattr(musicgen_model, '_model_id', '') != MODEL_ID:
    print(f'\n⏳ モデルをロード中: {MODEL_ID}')
    musicgen_model = MusicGen.get_pretrained(MODEL_ID)
    musicgen_model._model_id = MODEL_ID
    print(f'✅ モデルロード完了: {MODEL_ID}\n')
else:
    print(f'✅ モデルは既にロード済みです: {MODEL_ID}\n')

## Step 3: 生成関数の定義

| パラメータ | 型 | 説明 |
|-----------|-----|------|
| `descriptions` | `list[str]` | 生成する音楽の説明（英語推奨） |
| `duration` | `int` | 生成秒数（デフォルト: 10秒） |
| `melody_audio` | `tuple` | `(wave, sample_rate)` のタプル（任意） |
| `top_k` | `int` | サンプリングの多様性（高いほど多様） |
| `temperature` | `float` | ランダム性（高いほど自由） |

In [ ]:
# ── Step 3: 生成関数の定義 ────────────────────────────────────

import torchaudio
import numpy as np
from audiocraft.data.audio import audio_write

def generate_music(
    descriptions: list,
    duration: int = 10,
    melody_audio=None,
    top_k: int = 250,
    temperature: float = 1.0,
    guidance_scale: float = 3.0,
):
    """
    MusicGen で音楽を生成する。

    Returns:
        list of (wav_tensor [1, T], sample_rate) per description
    """
    musicgen_model.set_generation_params(
        duration        = duration,
        top_k           = top_k,
        temperature     = temperature,
        cfg_coef        = guidance_scale,
    )

    if melody_audio is not None:
        # melody_audio = (waveform_tensor [C, T], sample_rate)
        melody_wav, melody_sr = melody_audio
        # モノラル化・次元調整
        if melody_wav.dim() == 1:
            melody_wav = melody_wav.unsqueeze(0)
        melody_wav = melody_wav[:1]  # 最初のチャンネルのみ
        melody_wav = melody_wav.unsqueeze(0).to(device)  # [1, 1, T]
        wav = musicgen_model.generate_with_chroma(
            descriptions   = descriptions,
            melody_wavs    = melody_wav,
            melody_sample_rate = melody_sr,
            progress       = True,
        )
    else:
        wav = musicgen_model.generate(
            descriptions = descriptions,
            progress     = True,
        )

    sr = musicgen_model.sample_rate
    return [(wav[i].cpu(), sr) for i in range(len(descriptions))]

print('✅ generate_music() 関数の定義完了')

## Step 4: 生成と再生

プロンプトを英語で記述するほど精度が上がります。

**プロンプト例**:
```
lo-fi hip hop, chill beats, vinyl noise, relaxing, slow bpm
epic orchestral, rising tension, dramatic strings and brass
jazz piano trio, upbeat swing, walking bass, brushed drums
ambient synth pad, ethereal, slow evolving texture, cinematic
```

> `duration` は最大 30 秒が実用的です（small モデルは長いほど VRAM を消費します）。

In [ ]:
# ── Step 4: 生成と再生 ───────────────────────────────────────

import ipywidgets as widgets
from ipywidgets import (
    Textarea, IntSlider, FloatSlider, Dropdown,
    Button, VBox, HBox, Label, Output, HTML
)
from IPython.display import display, Audio
import datetime

# ─── ウィジェット ───────────────────────────────────────────
prompt_input = Textarea(
    value='lo-fi hip hop, chill beats, vinyl noise, relaxing, slow bpm',
    placeholder='音楽スタイルを英語で説明してください',
    description='プロンプト:',
    rows=3,
    style={'description_width': '100px'},
    layout=widgets.Layout(width='540px'),
)

duration_sl = IntSlider(
    value=10, min=5, max=30, step=1,
    description='生成秒数:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='420px'),
)

topk_sl = IntSlider(
    value=250, min=1, max=1000, step=50,
    description='Top-K:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='420px'),
)

temp_sl = FloatSlider(
    value=1.0, min=0.1, max=2.0, step=0.1,
    description='Temperature:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='420px'),
    readout_format='.1f',
)

cfg_sl = FloatSlider(
    value=3.0, min=1.0, max=10.0, step=0.5,
    description='CFG Scale:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='420px'),
    readout_format='.1f',
)

melody_path_input = widgets.Text(
    placeholder='（任意）参照メロディの WAV パス: /content/drive/MyDrive/...',
    description='メロディ:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='540px'),
)

generate_btn = Button(
    description='🎵 音楽を生成',
    button_style='success',
    layout=widgets.Layout(width='200px', height='42px'),
)

output_area = Output()

# ─── 生成ボタンのコールバック ──────────────────────────────
def on_generate(btn):
    output_area.clear_output()
    with output_area:
        prompt   = prompt_input.value.strip()
        duration = duration_sl.value

        if not prompt:
            print('❌ プロンプトを入力してください')
            return

        # メロディ（任意）のロード
        melody = None
        melody_path = melody_path_input.value.strip()
        if melody_path:
            if not os.path.exists(melody_path):
                print(f'⚠️  メロディファイルが見つかりません: {melody_path}')
                print('   メロディなしで続行します。')
            else:
                try:
                    m_wav, m_sr = torchaudio.load(melody_path)
                    melody = (m_wav, m_sr)
                    print(f'🎼 メロディ読込: {os.path.basename(melody_path)}')
                except Exception as e:
                    print(f'⚠️  メロディ読込失敗: {e}')

        print(f'⏳ 生成中 ...')
        print(f'  プロンプト : {prompt}')
        print(f'  生成秒数  : {duration}秒')
        print(f'  Top-K     : {topk_sl.value} / Temp: {temp_sl.value:.1f} / CFG: {cfg_sl.value:.1f}')

        try:
            results = generate_music(
                descriptions    = [prompt],
                duration        = duration,
                melody_audio    = melody,
                top_k           = topk_sl.value,
                temperature     = temp_sl.value,
                guidance_scale  = cfg_sl.value,
            )

            wav_tensor, sr = results[0]
            # [1, T] → [T] に変換してプレイヤーへ渡す
            wav_np = wav_tensor.squeeze().numpy()

            # ローカルに一時保存
            ts         = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
            local_path = f'{LOCAL_MUSIC_DIR}/musicgen_{ts}.wav'
            import soundfile as sf
            sf.write(local_path, wav_np, sr)

            # グローバルに保持（Step 5 の保存で参照）
            globals()['last_generated'] = {
                'wav_np' : wav_np,
                'sr'     : sr,
                'path'   : local_path,
                'prompt' : prompt,
                'ts'     : ts,
            }

            print(f'\n✅ 生成完了！  ({duration}秒 / {sr}Hz)')
            print(f'📁 ローカル保存: {local_path}')
            print(f'💡 Step 5 のセルで Google Drive に保存できます')
            display(Audio(wav_np, rate=sr, autoplay=False))

        except Exception as e:
            import traceback
            print(f'\n❌ エラー: {type(e).__name__}: {e}')
            traceback.print_exc()

generate_btn.on_click(on_generate)

# ─── レイアウト ──────────────────────────────────────────────
header = HTML(value="""
<div style='
    background: linear-gradient(135deg, #0d1117 0%, #1b2a4a 100%);
    color: #cdd9e5;
    padding: 12px 20px;
    border-radius: 6px 6px 0 0;
    font-family: monospace;
    font-size: 14px;
    letter-spacing: 1px;
'>
🎵 &nbsp; MusicGen 生成パネル
</div>
""")

panel = VBox([
    header,
    VBox([
        Label(value='📝 プロンプト'),
        prompt_input,
        melody_path_input,
        Label(value='⚙️ 生成パラメータ'),
        duration_sl,
        topk_sl,
        temp_sl,
        cfg_sl,
        generate_btn,
        output_area,
    ], layout=widgets.Layout(padding='16px', gap='10px')),
], layout=widgets.Layout(
    border='2px solid #1b2a4a',
    border_radius='8px',
    margin='10px 0',
))

display(panel)

## Step 5: Google Drive への保存

Step 4 で生成した音楽を `MyDrive/AI_Music/` に WAV 形式で書き出します。

In [ ]:
# ── Step 5: Google Drive に保存 ──────────────────────────────

import soundfile as sf
import shutil

def save_to_drive(result: dict = None):
    """
    生成済み音楽を Google Drive に WAV 形式で保存する。
    result が None の場合は last_generated を参照する。
    """
    data = result or globals().get('last_generated')
    if data is None:
        print('❌ 保存できる音楽がありません。Step 4 で生成してください。')
        return

    wav_np  = data['wav_np']
    sr      = data['sr']
    ts      = data['ts']
    prompt  = data['prompt']

    # ファイル名: timestamp_プロンプト先頭30文字（ファイル名安全化）
    safe_prompt = ''.join(c if c.isalnum() or c in '-_ ' else '_' for c in prompt[:30]).strip()
    filename    = f'{ts}_{safe_prompt}.wav'
    drive_path  = os.path.join(DRIVE_MUSIC_DIR, filename)

    sf.write(drive_path, wav_np, sr, subtype='PCM_16')

    size_mb = os.path.getsize(drive_path) / 1e6
    print(f'✅ Google Drive に保存しました')
    print(f'📁 パス   : {drive_path}')
    print(f'📊 サイズ : {size_mb:.2f} MB')
    print(f'🎵 プロンプト: {prompt}')
    return drive_path

# 実行
save_to_drive()

---

## (オプション) ComfyUI-AudioCraft のセットアップ

ComfyUI 上で MusicGen を使うためのカスタムノードをクローンします。

- このセルは ComfyUI 環境を別途用意している場合のみ実行してください。
- 通常の生成（Step 4）には不要です。

In [ ]:
# ── (オプション) ComfyUI-AudioCraft セットアップ ─────────────

import subprocess, sys

COMFYUI_ROOT       = '/content/ComfyUI'
CUSTOM_NODES_DIR   = f'{COMFYUI_ROOT}/custom_nodes'
AUDIOCRAFT_NODE    = 'ComfyUI-AudioCraft'
AUDIOCRAFT_REPO    = 'https://github.com/eigenpunk/ComfyUI-AudioCraft'

# ComfyUI のルートが存在しない場合はスキップ
if not os.path.isdir(COMFYUI_ROOT):
    print(f'⚠️  ComfyUI が見つかりません: {COMFYUI_ROOT}')
    print('   ComfyUI を先にインストールしてから再実行してください。')
else:
    node_dir = os.path.join(CUSTOM_NODES_DIR, AUDIOCRAFT_NODE)
    if os.path.isdir(node_dir):
        print(f'✅ {AUDIOCRAFT_NODE} は既にクローン済みです: {node_dir}')
    else:
        print(f'⏳ {AUDIOCRAFT_NODE} をクローン中...')
        result = subprocess.run(
            ['git', 'clone', AUDIOCRAFT_REPO, node_dir],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            print(f'✅ クローン完了: {node_dir}')
        else:
            print(f'❌ クローン失敗:\n{result.stderr}')

    # Python パスに追加
    for path in [COMFYUI_ROOT, CUSTOM_NODES_DIR]:
        if path not in sys.path:
            sys.path.insert(0, path)
            print(f'✅ sys.path に追加: {path}')

    print('\n📌 ComfyUI-AudioCraft のセットアップ完了')
    print('   ComfyUI を起動後、AudioCraft ノードが使用可能になります。')